# Mini Quantization · 从张量到模型

在普通 CPU 电脑上，用自己实现的 PyTorch helper 理解量化映射、重建误差和权重存储。

本 Notebook 由用户提供的 Python 导出脚本恢复并按学习主题归整。`helper.py` 保留原始实现；整理前脚本位于 `original/mini_quantization_export.py`。

**主线：** 非对称量化 → 对称量化 → 逐通道 / 分组量化 → 量化线性层 → 模型层替换。

CodeGen、DETR、OPT 和远端上传是可选章节，默认跳过。详细运行方式与待修复边界见 [项目说明](README.md)。文件中没有预填输出，未把未运行的模型结果标为已验证。


## 0. 环境与实验开关

先在本目录安装 `requirements.txt`。Windows 保留原脚本中先导入 `msvc_runtime` 的顺序；其他系统跳过。固定随机种子后，在同一次实验中保留输入，再改变一个参数做比较。


In [ ]:
import sys
if sys.platform == "win32":
    import msvc_runtime  # 保留原 Windows 环境的导入顺序
import torch
import importlib
import helper
from torch import nn
importlib.reload(helper)
import torch.nn.functional as F
from helper import (
    linear_q_with_scale_and_zero_point,
    linear_dequantization,
    get_q_scale_and_zero_point,
    linear_quantizaton,
    get_q_scale_symmetric,
    linear_q_symmetric,
    linear_q_symmetric_per_channel,
    linear_q_symmetric_per_group,
    linear_dequantization_per_group,
    plot_quantization_errors,
    quantized_linear_W8A32_without_bias,
    w8_a16_forward,
    W8A16LinearLayer,
    replace_linear_with_target,
    replace_linear_with_target_and_quantize,
    DummyModel,
    plot_detection_results,
)

from pathlib import Path
from IPython.display import display
torch.manual_seed(42)
print("PyTorch:", torch.__version__, "| 本实验使用 CPU")


In [ ]:
# 基础张量实验无需下载模型。只开启已经准备好本地缓存的模型。
RUN_CODEGEN = False
RUN_DETR = False
RUN_OPT = False
UPLOAD_TO_HF = False

IMAGE_PATH = "dinner.jpg"
ARTIFACT_DIR = Path("artifacts")
HF_REPO_ID = ""  # 可选上传时填写自己的目标模型仓库


## 1. 非对称线性量化（INT8）

- `scale = (r_max - r_min) / (q_max - q_min)`
- `zero_point = clamp(round(q_min - r_min / scale), q_min, q_max)`
- `q = clamp(round(x / scale + zero_point), q_min, q_max)`
- `x_hat = scale * (q - zero_point)`

这里沿用原 helper 的“先加零点，再取整”顺序。先观察随机输入，再在后续验收中补充全正、全负与常数输入。


In [ ]:
r_tensor = torch.randn((4, 4))
r_tensor


In [ ]:
quantized_tensor, scale, zero_point = linear_quantizaton(r_tensor)
print(f"scale:       {scale}")
print(f"zero_point:  {zero_point}")
print(f"quantized:\n{quantized_tensor}")


In [ ]:
dequantized_tensor = linear_dequantization(quantized_tensor, scale, zero_point)
dequantized_tensor


In [ ]:
(r_tensor - dequantized_tensor).square().mean()


In [ ]:
plot_quantization_errors(r_tensor, quantized_tensor, dequantized_tensor)


## 2. 对称线性量化（INT8）

`scale = max(abs(x)) / q_max`，`zero_point = 0`。原 helper 使用 dtype 的完整整数范围裁剪。

先观察随机数据，再观察一个大动态范围的小矩阵。比较 MSE 时也要看输入尺度，不能把两个不同输入的误差大小直接当成算法优劣。


In [ ]:
test_tensor = torch.randn((4, 4))
print(f"symmetric scale: {get_q_scale_symmetric(test_tensor)}")


In [ ]:
quantizated_tensor, scale = linear_q_symmetric(test_tensor)
quantizated_tensor


In [ ]:
dequantized_tensor = linear_dequantization(quantizated_tensor, scale, zero_point=0)
dequantized_tensor


In [ ]:
plot_quantization_errors(test_tensor, quantizated_tensor, dequantized_tensor)


In [ ]:
test_tensor = torch.tensor(
    [[191.6, -13.5, 728.6],
    [92.14, 295.5, -184],
    [0, 684.6, 245.5]]
)


In [ ]:
quantizated_tensor, scale = linear_q_symmetric(test_tensor)
dequantized_tensor = linear_dequantization(quantizated_tensor, scale, zero_point=0)
plot_quantization_errors(test_tensor, quantizated_tensor, dequantized_tensor)


## 3. 逐通道量化：行与列分别意味着什么？

固定同一个矩阵，对比全局 scale、每行 scale（`dim=0`）与每列 scale（`dim=1`）。

观察：scale 的形状怎样广播回原张量？误差改变的同时多存储了多少个 scale？结论以这组输入的实际结果为准。


In [ ]:
# 用大动态范围数据, 对比 per-tensor vs per-channel
test_tensor = torch.tensor(
    [[191.6, -13.5, 728.6],
    [92.14, 295.5, -184],
    [0, 684.6, 245.5]]
)

# per-tensor (全局一个 scale)
q_per_tensor, s_tensor = linear_q_symmetric(test_tensor)
dq_per_tensor = linear_dequantization(q_per_tensor, s_tensor, zero_point=0)
mse_per_tensor = (test_tensor - dq_per_tensor).square().mean()
print(f"Per-Tensor MSE:  {mse_per_tensor.item():.4f}")

# per-channel (每行独立 scale)
q_per_channel, s_channel = linear_q_symmetric_per_channel(test_tensor, dim=0)
dq_per_channel = linear_dequantization(q_per_channel, s_channel, zero_point=0)
mse_per_channel = (test_tensor - dq_per_channel).square().mean()
print(f"Per-Channel dim=0 MSE: {mse_per_channel.item():.4f}")

# per-channel (每列独立scale)
q_per_channel, s_channel = linear_q_symmetric_per_channel(test_tensor, dim=1)
dq_per_channel = linear_dequantization(q_per_channel, s_channel, zero_point=0)
mse_per_channel = (test_tensor - dq_per_channel).square().mean()
print(f"Per-Channel dim=1 MSE: {mse_per_channel.item():.4f}")


In [ ]:
plot_quantization_errors(test_tensor, q_per_channel, dq_per_channel)


## 4. 分组量化（Per-group）

原 helper 将二维张量按连续的 `group_size` 个元素拆分，每组独立计算 scale，再恢复原形状。

先运行 `group_size=6`，之后用同一个输入比较 1、2、3、6。记录误差、scale 个数，并说明哪些输入形状符合该实现的限制。


In [ ]:
test_tensor = torch.rand((6,6))


In [ ]:
group_size = 6


In [ ]:
quantized_tensor, scale = linear_q_symmetric_per_group(test_tensor, group_size)


In [ ]:
dequantized_tensor = linear_dequantization_per_group(quantized_tensor, scale, group_size)


In [ ]:
plot_quantization_errors(test_tensor, quantized_tensor, dequantized_tensor)


## 5. W8A32 线性层：权重量化后输出差多少？

激活为 FP32，权重先做 INT8 对称量化，再反量化调用浮点线性运算。对照同一个输入下的原始 FP32 输出。

本示例 `z_w=0`。helper 的该函数目前没有正确处理非零 zero-point，已列为待修复项。


In [ ]:
input = torch.tensor([1, 2, 3], dtype = torch.float32)


In [ ]:
weight = torch.tensor([[-2, -1.13, 0.42],
                      [-1.51, 0.25, 1.62],
                      [0.12, 1.35, 2.15]])


In [ ]:
q_w, s_w = linear_q_symmetric(weight)


In [ ]:
q_w


In [ ]:
s_w


In [ ]:
output = quantized_linear_W8A32_without_bias(input, q_w, s_w, 0)


In [ ]:
print(f"This is the W8A32 output: {output}")


In [ ]:
fp32_output =  torch.nn.functional.linear(input, weight)


In [ ]:
print(f"This is the output if we don't quantize: {fp32_output}")


## 6. 量化线性层：存储 dtype 与计算 dtype

类名 `W8A16LinearLayer` 沿用原实现，但实际输入 dtype 决定浮点计算类型。本例权重为 BF16，不能把它写成 FP16 激活的实验结果。

整理时把模块的形状声明改为 `in_features=8, out_features=4`，与原脚本的 `(4, 8)` 权重一致。这里先观察权重重建误差，尚未验证真实模型的速度收益。


In [ ]:
module = W8A16LinearLayer(in_features=8, out_features=4, dtype=torch.bfloat16)


In [ ]:
random_matrix = torch.randn((4, 8), dtype = torch.bfloat16)


In [ ]:
random_matrix


In [ ]:
module.quantize(random_matrix)


In [ ]:
module.int8_weights


In [ ]:
module.scales.shape


In [ ]:
module.int8_weights.shape


In [ ]:
dequantized_weights = module.int8_weights*module.scales.unsqueeze(1)


In [ ]:
dequantized_weights


In [ ]:
(random_matrix - dequantized_weights).square().mean()


## 7. 递归替换模型中的 Linear

先看“只替换结构”，再看“替换并量化原权重”。`DummyModel` 中保留 `lm_head`，检查打印出的层结构。

原实现按叶子模块名匹配排除项，而不是完整层路径。真实模型使用 `"0"`、`"1"` 等排除项时，应先确认它们实际匹配了哪些位置。


In [ ]:
model_1 = DummyModel()
model_2 = DummyModel()


In [ ]:
replace_linear_with_target(model_1, W8A16LinearLayer, ["lm_head"])


In [ ]:
print(model_1)


In [ ]:
model_3 = DummyModel()


In [ ]:
replace_linear_with_target_and_quantize(model_3, W8A16LinearLayer, ["lm_head"])
print(model_3)


## 8. 可选：CodeGen 文本生成前后对比

需要额外模型依赖与本地 `Salesforce/codegen-350M-mono` 缓存。设置 `RUN_CODEGEN=True` 后运行。

这是输出观察示例，没有建立准确率或性能评估集。模型和 tokenizer 保持 `local_files_only=True`。


In [ ]:
if RUN_CODEGEN:
    from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline
    model_id = "Salesforce/codegen-350M-mono"
    model = AutoModelForCausalLM.from_pretrained(model_id, torch_dtype = torch.bfloat16, low_cpu_mem_usage = True, local_files_only=True)
    tokenizer = AutoTokenizer.from_pretrained(model_id, local_files_only=True)

    pipe = pipeline("text-generation", model = model, tokenizer = tokenizer)

    print(pipe("def hello_word():", max_new_tokens = 20, do_sample = False))

    replace_linear_with_target_and_quantize(model, W8A16LinearLayer, ["lm_head"])

    print(pipe("def hello_word():", max_new_tokens = 20, do_sample = False)[0]["generated_text"])
else:
    print("跳过 CodeGen：需要时在环境配置中开启 RUN_CODEGEN。")


## 9. 可选：DETR 目标检测与权重占用观察

需要额外模型依赖、本地 `facebook/detr-resnet-50` 缓存，以及 `IMAGE_PATH` 指向的图片。

原示例展示量化前的检测标注和替换前后的模型参数 / buffer 占用差异，尚未比较量化后的检测准确率、延迟或峰值内存。


In [ ]:
if RUN_DETR:
    from transformers import AutoImageProcessor, AutoModelForObjectDetection
    from PIL import Image
    import timm
    model_id = "facebook/detr-resnet-50"

    processor = AutoImageProcessor.from_pretrained(model_id, local_files_only=True)
    model = AutoModelForObjectDetection.from_pretrained(model_id, local_files_only=True)

    previous_memory_footprint = model.get_memory_footprint()
    display(previous_memory_footprint)

    image = Image.open(IMAGE_PATH)
    inputs = processor(images = image, return_tensors = "pt")
    with torch.no_grad():
        outputs = model(**inputs)
    results = processor.post_process_object_detection(
        outputs, threshold=0.5, target_sizes=torch.tensor([image.size[::-1]])
      )
    annotated_image = plot_detection_results(image, results, model.config.id2label, threshold=0.5)
    display(annotated_image)

    replace_linear_with_target_and_quantize(model, W8A16LinearLayer, ["0", "1", "2", "class_labels_classifier"])

    new_memory_footprint = model.get_memory_footprint()

    display(new_memory_footprint)

    display(previous_memory_footprint - new_memory_footprint)
else:
    print("跳过 DETR：需要时在环境配置中开启 RUN_DETR。")


## 10. 可选：OPT 权重量化与序列化

需要本地 `facebook/opt-125m` 缓存。设置 `RUN_OPT=True` 后，将 state_dict 保存到 `artifacts/quantized_state_dict.pth`。

恢复时还需要构建匹配的量化层结构和配置；仅保存这个文件不代表已经完成 Transformers 模型的打包与恢复验证。


In [ ]:
if RUN_OPT:
    import torch
    from transformers import AutoModelForCausalLM, AutoTokenizer

    model_id = "facebook/opt-125m"

    tokenizer = AutoTokenizer.from_pretrained(model_id, local_files_only=True)
    model = AutoModelForCausalLM.from_pretrained(
          model_id,
          dtype=torch.float32,        # 125M 小模型用 FP32 即可
          local_files_only=True,      # 跳过网络，用本地缓存
      )

    replace_linear_with_target_and_quantize(model, W8A16LinearLayer, ["lm_head"])

    quantized_state_dict = model.state_dict()
    ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
    torch.save(quantized_state_dict, ARTIFACT_DIR / "quantized_state_dict.pth")
    print("已保存:", ARTIFACT_DIR / "quantized_state_dict.pth")
else:
    print("跳过 OPT：需要时在环境配置中开启 RUN_OPT。")


## 11. 可选：手动上传到 Hugging Face

先完成 OPT 实验，准备好目标模型仓库并在本地认证，再填写 `HF_REPO_ID`，最后将 `UPLOAD_TO_HF=True`。本单元默认跳过。


In [ ]:
if UPLOAD_TO_HF:
    if not RUN_OPT:
        raise ValueError("请先开启 RUN_OPT 并完成权重保存。")
    if not HF_REPO_ID:
        raise ValueError("请先填写自己的 HF_REPO_ID。")
    from huggingface_hub import HfApi
    weight_path = ARTIFACT_DIR / "quantized_state_dict.pth"
    if not weight_path.is_file():
        raise FileNotFoundError(weight_path)
    HfApi().upload_file(
        path_or_fileobj=str(weight_path),
        path_in_repo="quantized_state_dict.pth",
        repo_id=HF_REPO_ID,
        repo_type="model",
    )
else:
    print("跳过 Hugging Face 上传；需要时显式开启 UPLOAD_TO_HF。")


## 12. 阶段复盘与验收

| 方法 | 主要参数 | 本次要观察的取舍 |
| --- | --- | --- |
| 非对称量化 | scale、zero-point | 表示范围如何随偏移改变，是否发生裁剪 |
| 对称量化 | scale、zero-point=0 | 对称范围与离群值的影响 |
| 逐通道量化 | 每通道一个 scale | 不同通道的量级差异与额外元数据 |
| 分组量化 | group_size、每组 scale | 更细的粒度与 scale 存储开销 |

请用自己的话解释：

1. 舍入误差与裁剪误差分别在什么位置产生？
2. 为什么同一位宽下，scale 越小不一定让 MSE 越小？
3. 为什么 INT8 权重文件变小不能直接证明推理加速？
4. 这个 helper 对全零、常数张量和非零 zero-point 还缺哪些验证？

使用 [实验记录模板](experiments/TEMPLATE.md) 保存一组固定输入的对照实验。完成实际验证后再更新 [README 中的验收清单](README.md#已知问题与下一步验收)。
